# Q1: Fuzzy Logic System — Washing Machine Cycle Time
**Inputs:** Dirt Level, Load Size  
**Output:** Cycle Time (minutes)  
**Tool:** scikit-fuzzy

In [ ]:
# Step 1: Install scikit-fuzzy
!pip install scikit-fuzzy

In [2]:
# Step 2: Import Libraries
import numpy as np
import skfuzzy as fuzz
from skfuzzy import control as ctrl
import matplotlib.pyplot as plt

In [3]:
# Step 3: Define Universe of Discourse (input/output ranges)
dirt_level  = ctrl.Antecedent(np.arange(0, 11, 1), 'dirt_level')   # 0 to 10
load_size   = ctrl.Antecedent(np.arange(0, 11, 1), 'load_size')    # 0 to 10
cycle_time  = ctrl.Consequent(np.arange(0, 91, 1), 'cycle_time')   # 0 to 90 minutes

In [4]:
# Step 4: Define Membership Functions

# --- Dirt Level: Low, Medium, High ---
dirt_level['low']    = fuzz.trimf(dirt_level.universe,  [0, 0, 5])
dirt_level['medium'] = fuzz.trimf(dirt_level.universe,  [2, 5, 8])
dirt_level['high']   = fuzz.trimf(dirt_level.universe,  [5, 10, 10])

# --- Load Size: Small, Medium, Large ---
load_size['small']   = fuzz.trimf(load_size.universe,   [0, 0, 5])
load_size['medium']  = fuzz.trimf(load_size.universe,   [2, 5, 8])
load_size['large']   = fuzz.trimf(load_size.universe,   [5, 10, 10])

# --- Cycle Time: Short, Medium, Long ---
cycle_time['short']  = fuzz.trimf(cycle_time.universe,  [0, 0, 40])
cycle_time['medium'] = fuzz.trimf(cycle_time.universe,  [20, 45, 70])
cycle_time['long']   = fuzz.trimf(cycle_time.universe,  [50, 90, 90])

In [ ]:
# Step 5: Plot Membership Functions
fig, axes = plt.subplots(nrows=1, ncols=3, figsize=(16, 4))

dirt_level.view(ax=axes[0])
axes[0].set_title('Dirt Level Membership Functions')

load_size.view(ax=axes[1])
axes[1].set_title('Load Size Membership Functions')

cycle_time.view(ax=axes[2])
axes[2].set_title('Cycle Time Membership Functions')

plt.tight_layout()
plt.savefig('membership_functions.png', dpi=150)
plt.show()
print('Membership function plot saved!')

In [ ]:
# Step 6: Define Fuzzy Rules (at least 6)
rule1 = ctrl.Rule(dirt_level['low']    & load_size['small'],  cycle_time['short'])
rule2 = ctrl.Rule(dirt_level['low']    & load_size['medium'], cycle_time['short'])
rule3 = ctrl.Rule(dirt_level['low']    & load_size['large'],  cycle_time['medium'])
rule4 = ctrl.Rule(dirt_level['medium'] & load_size['small'],  cycle_time['short'])
rule5 = ctrl.Rule(dirt_level['medium'] & load_size['medium'], cycle_time['medium'])
rule6 = ctrl.Rule(dirt_level['medium'] & load_size['large'],  cycle_time['medium'])
rule7 = ctrl.Rule(dirt_level['high']   & load_size['small'],  cycle_time['medium'])
rule8 = ctrl.Rule(dirt_level['high']   & load_size['medium'], cycle_time['long'])
rule9 = ctrl.Rule(dirt_level['high']   & load_size['large'],  cycle_time['long'])

print('9 fuzzy rules defined successfully!')

In [7]:
# Step 7: Build the Fuzzy Control System
washing_ctrl = ctrl.ControlSystem([
    rule1, rule2, rule3,
    rule4, rule5, rule6,
    rule7, rule8, rule9
])
washing_sim = ctrl.ControlSystemSimulation(washing_ctrl)

In [ ]:
# Step 8 Sample Inputs to test the system and user inputs also allowed
def get_cycle_time(dirt, load, label):
    if dirt < 0 or dirt > 10:
        print("Error: Dirt Level must be between 0 and 10!")
        return
    if load < 0 or load > 10:
        print("Error: Load Size must be between 0 and 10!")
        return

    washing_sim.input['dirt_level'] = dirt
    washing_sim.input['load_size']  = load
    washing_sim.compute()
    result = washing_sim.output['cycle_time']
    print(f"{label:<30} {dirt:>6} {load:>6} {result:>18.2f}")

# Test Cases
test_cases = [
    (2, 2,  'Low dirt, Small load'),
    (5, 5,  'Medium dirt, Medium load'),
    (8, 8,  'High dirt, Large load'),
    (3, 7,  'Low dirt, Large load'),
    (7, 3,  'High dirt, Small load'),
]

print(f"{'Test Case':<30} {'Dirt':>6} {'Load':>6} {'Cycle Time (min)':>18}")
print('_' * 63)
for dirt, load, label in test_cases:
    get_cycle_time(dirt, load, label)

# User Input
print("\n")
print("Now enter your own values!")
print("=" * 28)
print("Dirt Level Scale:")
print("  0-3  → Low")
print("  4-6  → Medium")
print("  7-10 → High")
print("=" * 28)
print("Load Size Scale:")
print("  0-3  → Small")
print("  4-6  → Medium")
print("  7-10 → Large")
print("=" * 28)

dirt = float(input("Enter Dirt Level (0-10): "))
load = float(input("Enter Load Size (0-10): "))
get_cycle_time(dirt, load, "User Input")


In [ ]:
# Step 9: Rule Viewer — Heatmap
import seaborn as sns

dirt_range = np.arange(0, 11, 1)
load_range = np.arange(0, 11, 1)
Z = np.zeros((len(dirt_range), len(load_range)))

for i, d in enumerate(dirt_range):
    for j, l in enumerate(load_range):
        washing_sim.input['dirt_level'] = d
        washing_sim.input['load_size']  = l
        washing_sim.compute()
        Z[i, j] = washing_sim.output['cycle_time']

plt.figure(figsize=(8, 6))
sns.heatmap(Z,
            xticklabels=load_range,
            yticklabels=dirt_range,
            cmap='YlOrRd',
            annot=True,
            fmt='.0f',
            cbar_kws={'label': 'Cycle Time (minutes)'})

plt.xlabel('Load Size')
plt.ylabel('Dirt Level')
plt.title('Fuzzy Rule Viewer — Cycle Time Heatmap')
plt.tight_layout()
plt.show()